# Version 9 : Cox Proportional Hazards - Optimisation Complète

**Modèle** : Cox Proportional Hazards (CoxPH)
- Modèle linéaire pour analyse de survie
- Excellent pour capturer des relations linéaires
- Complémentaire aux modèles non-linéaires (RSF, XGBoost)

**Optimisations** :
1. 📊 Feature Selection (variance, corrélation, importance)
2. 🎯 Regularization (L1, L2, ElasticNet)
3. ⚙️ Hyperparameter Tuning (Optuna)
4. 🔄 Cross-Validation

**Objectif** : C-index > 0.73

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Data Loading & Feature Engineering

In [2]:
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
clinical_test = pd.read_csv(f"{DATA_PATH}\\X_test\\clinical_test.csv")
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")
molecular_test = pd.read_csv(f"{DATA_PATH}\\X_test\\molecular_test.csv")

print("✓ Data loaded")

✓ Data loaded


In [3]:
# Feature engineering (same as V8.1)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def create_advanced_features(X_df):
    X_adv = X_df.copy()
    X_adv['blast_to_wbc'] = X_adv['BM_BLAST'] / (X_adv['WBC'] + 1)
    X_adv['monocyte_ratio'] = X_adv['MONOCYTES'] / (X_adv['WBC'] + 1)
    X_adv['anc_ratio'] = X_adv['ANC'] / (X_adv['WBC'] + 1)
    X_adv['platelet_to_blast'] = X_adv['PLT'] / (X_adv['BM_BLAST'] + 1)
    X_adv['hb_to_plt'] = X_adv['HB'] / (X_adv['PLT'] + 1)
    X_adv['vaf_mutation_burden'] = X_adv['vaf_mean'] * X_adv['mutation_count_total']
    X_adv['vaf_per_mutation'] = X_adv['vaf_sum'] / (X_adv['mutation_count_total'] + 1)
    X_adv['cytogenetic_risk_score'] = (
        X_adv['cyto_complex'] * 3 + X_adv['cyto_chr7_affected'] * 2 + 
        X_adv['cyto_del5q'] * 1.5 + X_adv['cyto_loss_count'] * 0.5
    )
    X_adv['blast_cytogenetic_risk'] = X_adv['BM_BLAST'] * X_adv['cytogenetic_risk_score']
    X_adv['blast_to_mutation'] = X_adv['BM_BLAST'] / (X_adv['mutation_count_total'] + 1)
    X_adv['wbc_plt_index'] = X_adv['WBC'] * X_adv['PLT'] / 1000
    X_adv['blast_hb_ratio'] = X_adv['BM_BLAST'] / (X_adv['HB'] + 1)
    X_adv['monocyte_blast_ratio'] = X_adv['MONOCYTES'] / (X_adv['BM_BLAST'] + 1)
    X_adv['mutation_per_vaf'] = X_adv['mutation_count_total'] / (X_adv['vaf_mean'] + 0.01)
    X_adv['cyto_anomaly_density'] = X_adv['cyto_total_anomalies'] / (X_adv['cyto_total_anomalies'].max() + 1)
    X_adv['blast_mutation_interaction'] = X_adv['BM_BLAST'] * X_adv['mutation_count_total']
    X_adv['blast_vaf_interaction'] = X_adv['BM_BLAST'] * X_adv['vaf_mean']
    X_adv['blast_cyto_complex'] = X_adv['BM_BLAST'] * X_adv['cyto_complex']
    X_adv['tp53_blast'] = X_adv['gene_TP53_present'] * X_adv['BM_BLAST']
    X_adv['runx1_mutation_burden'] = X_adv['gene_RUNX1_count'] * X_adv['mutation_count_total']
    X_adv['nras_vaf'] = X_adv['gene_NRAS_present'] * X_adv['vaf_mean']
    X_adv['cyto_mutation_interaction'] = X_adv['cyto_total_anomalies'] * X_adv['mutation_count_total']
    X_adv['chr7_blast'] = X_adv['cyto_chr7_affected'] * X_adv['BM_BLAST']
    X_adv['vaf_cyto_burden'] = X_adv['vaf_sum'] * X_adv['cyto_total_anomalies']
    X_adv['vaf_tp53'] = X_adv['vaf_mean'] * X_adv['gene_TP53_present']
    X_adv['log_wbc'] = np.log1p(X_adv['WBC'])
    X_adv['log_plt'] = np.log1p(X_adv['PLT'])
    X_adv['log_blast'] = np.log1p(X_adv['BM_BLAST'])
    X_adv['log_mutation_count'] = np.log1p(X_adv['mutation_count_total'])
    X_adv['log_vaf_sum'] = np.log1p(X_adv['vaf_sum'])
    return X_adv

# Create features
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids)

target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical.index, fill_value=0)
X_all_base = pd.concat([X_clinical, mol_features_train_aligned, cyto_features_train_aligned], axis=1)
X_all = create_advanced_features(X_all_base)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Features created: {X_all.shape[1]} total")

✓ Features created: 162 total


## 3. Preprocessing (Critical for CoxPH)

In [4]:
print("="*60)
print("PREPROCESSING FOR COXPH")
print("="*60)

# Imputation
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_all),
    index=X_all.index,
    columns=X_all.columns
)

# Remove zero variance features
selector = VarianceThreshold(threshold=0.01)
X_var = pd.DataFrame(
    selector.fit_transform(X_imputed),
    index=X_imputed.index,
    columns=X_imputed.columns[selector.get_support()]
)

# Standardization (CRITICAL for CoxPH)
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_var),
    index=X_var.index,
    columns=X_var.columns
)

print(f"\n✓ Preprocessing complete:")
print(f"  Original features:      {X_all.shape[1]}")
print(f"  After variance filter:  {X_var.shape[1]}")
print(f"  Standardized:           {X_scaled.shape[1]}")

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_surv, test_size=0.3, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"\n✓ Split: {len(X_train)} train, {len(X_val)} val")
print("="*60)

PREPROCESSING FOR COXPH

✓ Preprocessing complete:
  Original features:      162
  After variance filter:  147
  Standardized:           147

✓ Split: 2221 train, 952 val


## 4. Baseline CoxPH

In [5]:
print("="*60)
print("BASELINE: COXPH (L2 REGULARIZATION)")
print("="*60)

# Test différents alphas
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
results_baseline = []

for alpha in alphas:
    cox = CoxPHSurvivalAnalysis(alpha=alpha)
    cox.fit(X_train, y_train)
    
    y_pred_val = cox.predict(X_val)
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_val
    )[0]
    
    results_baseline.append({'alpha': alpha, 'C-index': c_index})
    print(f"  alpha={alpha:6.3f} → C-index: {c_index:.4f}")

best_baseline = max(results_baseline, key=lambda x: x['C-index'])
print(f"\n✓ Best baseline: alpha={best_baseline['alpha']}, C-index={best_baseline['C-index']:.4f}")

BASELINE: COXPH (L2 REGULARIZATION)
  alpha= 0.001 → C-index: 0.7313
  alpha= 0.010 → C-index: 0.7313
  alpha= 0.100 → C-index: 0.7314
  alpha= 1.000 → C-index: 0.7325
  alpha=10.000 → C-index: 0.7342

✓ Best baseline: alpha=10.0, C-index=0.7342


## 5. Feature Selection with CoxNet (L1 Regularization)

In [7]:
print("="*60)
print("FEATURE SELECTION WITH COXNET (LASSO)")
print("="*60)

# CoxNet with L1 (Lasso) for feature selection
coxnet = CoxnetSurvivalAnalysis(
    l1_ratio=1.0,  # Pure L1 (Lasso)
    fit_baseline_model=True
)
coxnet.fit(X_train, y_train)

# Get coefficients at different alphas (TRANSPOSE!)
coefficients = pd.DataFrame(
    coxnet.coef_.T,  # ← AJOUT DU .T pour transpose
    index=coxnet.alphas_,
    columns=X_train.columns
)

# Find best alpha
c_indices = []
for alpha_idx, alpha in enumerate(coxnet.alphas_):
    # Get predictions with this alpha
    y_pred = coxnet.predict(X_val, alpha=alpha)
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred
    )[0]
    c_indices.append(c_index)

best_alpha_idx = np.argmax(c_indices)
best_alpha_lasso = coxnet.alphas_[best_alpha_idx]
best_c_index_lasso = c_indices[best_alpha_idx]

# Get selected features (non-zero coefficients)
selected_coefs = coefficients.loc[best_alpha_lasso]
selected_features = selected_coefs[selected_coefs != 0].index.tolist()

print(f"\n✓ Best Lasso alpha: {best_alpha_lasso:.6f}")
print(f"✓ C-index: {best_c_index_lasso:.4f}")
print(f"✓ Selected features: {len(selected_features)} / {X_train.shape[1]}")
print(f"\n  Top 20 by absolute coefficient:")
top_features = selected_coefs[selected_coefs != 0].abs().sort_values(ascending=False).head(20)
print(top_features.to_string())

FEATURE SELECTION WITH COXNET (LASSO)

✓ Best Lasso alpha: 0.009682
✓ C-index: 0.7381
✓ Selected features: 69 / 147

  Top 20 by absolute coefficient:
HB                             0.252226
log_mutation_count             0.216216
blast_to_mutation              0.170100
cyto_normal                    0.158471
log_plt                        0.154249
gene_TP53_count                0.132554
vaf_tp53                       0.123780
effect_PTD                     0.104857
gene_ASXL1_present             0.090764
vaf_mutation_burden            0.086879
CENTER_TUD                     0.082743
gene_RUNX1_present             0.078028
CENTER_MUV                     0.076980
log_blast                      0.071687
PLT                            0.066530
cyto_has_loss                  0.061787
effect_non_synonymous_codon    0.058220
CENTER_PV                      0.054685
vaf_max                        0.051987
cyto_del_count                 0.050532


## 6. Test Different Feature Set Sizes

In [8]:
print("="*60)
print("TESTING DIFFERENT FEATURE SET SIZES")
print("="*60)

# Get features ordered by abs(coefficient)
feature_importance = selected_coefs[selected_coefs != 0].abs().sort_values(ascending=False)

# Test different numbers of features
n_features_to_test = [20, 30, 50, 75, 100, len(selected_features)]
results_feature_selection = []

for n_features in n_features_to_test:
    if n_features > len(feature_importance):
        n_features = len(feature_importance)
    
    selected_n = feature_importance.head(n_features).index.tolist()
    
    X_train_sel = X_train[selected_n]
    X_val_sel = X_val[selected_n]
    
    # Train with best baseline alpha
    cox = CoxPHSurvivalAnalysis(alpha=best_baseline['alpha'])
    cox.fit(X_train_sel, y_train)
    
    y_pred = cox.predict(X_val_sel)
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred
    )[0]
    
    results_feature_selection.append({
        'n_features': n_features,
        'C-index': c_index
    })
    print(f"  {n_features:3d} features → C-index: {c_index:.4f}")

best_n_features = max(results_feature_selection, key=lambda x: x['C-index'])
print(f"\n✓ Best: {best_n_features['n_features']} features, C-index: {best_n_features['C-index']:.4f}")

TESTING DIFFERENT FEATURE SET SIZES
   20 features → C-index: 0.7314
   30 features → C-index: 0.7338
   50 features → C-index: 0.7354
   69 features → C-index: 0.7362
   69 features → C-index: 0.7362
   69 features → C-index: 0.7362

✓ Best: 69 features, C-index: 0.7362


## 7. Hyperparameter Optimization with Optuna

In [9]:
print("="*60)
print("HYPERPARAMETER OPTIMIZATION (OPTUNA)")
print("="*60)

# Use best number of features
best_features = feature_importance.head(best_n_features['n_features']).index.tolist()
X_train_opt = X_train[best_features]
X_val_opt = X_val[best_features]

def objective(trial):
    # Try both CoxPH and CoxNet
    model_type = trial.suggest_categorical('model_type', ['CoxPH', 'CoxNet'])
    
    if model_type == 'CoxPH':
        alpha = trial.suggest_float('alpha', 1e-5, 100, log=True)
        model = CoxPHSurvivalAnalysis(alpha=alpha)
        model.fit(X_train_opt, y_train)
        y_pred = model.predict(X_val_opt)
    else:  # CoxNet
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        model = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, fit_baseline_model=True)
        model.fit(X_train_opt, y_train)
        # Use best alpha from cross-validation
        y_pred = model.predict(X_val_opt)
    
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred
    )[0]
    
    return c_index

print(f"\nOptimizing with {best_n_features['n_features']} features...")
print("Running 100 trials...\n")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
best_c_index_opt = study.best_value

print(f"\n✓ Optimization complete")
print(f"\nBest hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")
print(f"\nBest C-index: {best_c_index_opt:.4f}")

HYPERPARAMETER OPTIMIZATION (OPTUNA)

Optimizing with 69 features...
Running 100 trials...



  0%|          | 0/100 [00:00<?, ?it/s]


✓ Optimization complete

Best hyperparameters:
  model_type: CoxNet
  l1_ratio: 0.9025866771499649

Best C-index: 0.7371


## 8. Final Optimized Model

In [10]:
print("="*60)
print("FINAL OPTIMIZED MODEL")
print("="*60)

# Train final model with best params
if best_params['model_type'] == 'CoxPH':
    final_model = CoxPHSurvivalAnalysis(alpha=best_params['alpha'])
else:
    final_model = CoxnetSurvivalAnalysis(
        l1_ratio=best_params['l1_ratio'], 
        fit_baseline_model=True
    )

final_model.fit(X_train_opt, y_train)
y_pred_final = final_model.predict(X_val_opt)

c_index_final = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_final
)[0]

print(f"\nFinal C-index: {c_index_final:.4f}")

FINAL OPTIMIZED MODEL

Final C-index: 0.7371


## 9. Performance Comparison

In [ ]:
print("="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

comparison = pd.DataFrame([
    {'Model': 'V4 RSF Baseline', 'Features': 90, 'C-index': 0.7404},
    {'Model': 'V8.1 Ensemble', 'Features': 'Mixed', 'C-index': 0.7427},
    {'Model': 'CoxPH Baseline', 'Features': X_scaled.shape[1], 'C-index': best_baseline['C-index']},
    {'Model': 'CoxNet Lasso', 'Features': len(selected_features), 'C-index': best_c_index_lasso},
    {'Model': f'CoxPH {best_n_features["n_features"]} features', 'Features': best_n_features['n_features'], 'C-index': best_n_features['C-index']},
    {'Model': 'V9 CoxPH Optimized', 'Features': best_n_features['n_features'], 'C-index': c_index_final}
])

print("\n" + comparison.to_string(index=False))

improvement = c_index_final - 0.7404

print(f"\n📊 Improvement vs V4: {improvement:+.4f}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

models = ['V4\nRSF', 'V8.1\nEnsemble', 'CoxPH\nBaseline', 'CoxNet\nLasso', 'CoxPH\nSelected', 'V9\nOptimized']
scores = [
    0.7404, 0.7427, 
    best_baseline['C-index'], 
    best_c_index_lasso, 
    best_n_features['C-index'], 
    c_index_final
]
colors = ['#808080', '#27AE60', '#2E86AB', '#F18F01', '#A23B72', '#E74C3C']

bars = ax.bar(models, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(y=0.74, color='orange', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
ax.axhline(y=0.75, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Target: 0.75')

ax.set_ylabel('C-index (Validation)', fontsize=12)
ax.set_title('CoxPH Optimization Progress', fontsize=14, fontweight='bold')
ax.set_ylim([0.70, max(0.76, c_index_final + 0.01)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Test Set & Submission

In [ ]:
print("="*60)
print("PREPARING TEST SET")
print("="*60)

# Create features for test (same pipeline)
cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids)

clinical_test_indexed = clinical_test.set_index('ID')
X_numeric_test = clinical_test_indexed[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test_indexed['CENTER'], prefix='CENTER', drop_first=True)
for col in center_encoded.columns:
    if col not in center_encoded_test.columns:
        center_encoded_test[col] = 0
center_encoded_test = center_encoded_test[center_encoded.columns]
X_clinical_test = pd.concat([X_numeric_test, center_encoded_test], axis=1)

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)
X_test_all_base = pd.concat([X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned], axis=1)
X_test_all = create_advanced_features(X_test_all_base)

# Apply same preprocessing
for col in X_all.columns:
    if col not in X_test_all.columns:
        X_test_all[col] = 0
X_test_all = X_test_all[X_all.columns]

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_all),
    index=X_test_all.index,
    columns=X_test_all.columns
)

X_test_var = pd.DataFrame(
    selector.transform(X_test_imputed),
    index=X_test_imputed.index,
    columns=X_imputed.columns[selector.get_support()]
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_var),
    index=X_test_var.index,
    columns=X_test_var.columns
)

X_test_opt = X_test_scaled[best_features]

print(f"✓ Test set prepared: {X_test_opt.shape}")

In [ ]:
print("\nRetraining on full dataset...")

# Full training set
X_full_opt = X_scaled[best_features]

if best_params['model_type'] == 'CoxPH':
    model_full = CoxPHSurvivalAnalysis(alpha=best_params['alpha'])
else:
    model_full = CoxnetSurvivalAnalysis(
        l1_ratio=best_params['l1_ratio'],
        fit_baseline_model=True
    )

model_full.fit(X_full_opt, y_surv)

print("✓ Model retrained")

In [ ]:
# Generate predictions
predictions_test = model_full.predict(X_test_opt)

# Convert to risk scores
min_pred = predictions_test.min()
max_pred = predictions_test.max()
risk_scores = 1 - (predictions_test - min_pred) / (max_pred - min_pred)

submission = pd.DataFrame({
    'ID': X_test_opt.index,
    'risk_score': risk_scores
})

submission_path = f"{DATA_PATH}\\submission_v9_coxph_optimized.csv"
submission.to_csv(submission_path, index=False)

print("="*60)
print("SUBMISSION GENERATED")
print("="*60)
print(f"File: {submission_path}")
print(f"Predictions: {len(submission)}")
print(f"\n📊 Risk Scores (0-1):")
print(submission['risk_score'].describe())

## 11. Summary

In [ ]:
print("="*60)
print("VERSION 9 - COXPH OPTIMIZATION SUMMARY")
print("="*60)

print("\n🎯 OPTIMIZATION STEPS:")
print(f"  1. Variance filtering:  {X_all.shape[1]} → {X_var.shape[1]} features")
print(f"  2. Lasso selection:     → {len(selected_features)} features")
print(f"  3. Size optimization:   → {best_n_features['n_features']} features")
print(f"  4. Hyperparameter tuning")

print("\n📊 RESULTS:")
print(f"  Baseline (all features): {best_baseline['C-index']:.4f}")
print(f"  Lasso selection:         {best_c_index_lasso:.4f}")
print(f"  Best feature size:       {best_n_features['C-index']:.4f}")
print(f"  Final optimized:         {c_index_final:.4f}")

print("\n🚀 IMPROVEMENT:")
print(f"  vs V4 RSF:      {improvement:+.4f}")
print(f"  vs Baseline:    {c_index_final - best_baseline['C-index']:+.4f}")

print("\n⚙️ FINAL CONFIGURATION:")
print(f"  Model type:     {best_params['model_type']}")
print(f"  Features:       {best_n_features['n_features']}")
if best_params['model_type'] == 'CoxPH':
    print(f"  Alpha (L2):     {best_params['alpha']:.6f}")
else:
    print(f"  L1 ratio:       {best_params['l1_ratio']:.3f}")

if c_index_final >= 0.75:
    print("\n🎯 TARGET REACHED! (C-index ≥ 0.75)")
elif c_index_final > 0.74:
    print("\n✅ IMPROVED vs Baseline!")

print("\n✅ OUTPUT:")
print(f"  Submission: submission_v9_coxph_optimized.csv")
print(f"  Ready for V10 ensemble (RSF + XGBoost + CoxPH)!")